# Preparation

## Load base models and apply LoRA

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Running on: {device}")

model_id = "Qwen/Qwen2.5-1.5B-Instruct" 
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# Load in bfloat16 to save memory
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16
).to(device)

# Apply LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

Running on: mps


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705


## Prepare the Datasets

In [2]:
from datasets import load_dataset

math_data = load_dataset("openai/gsm8k", "main", split="train[:50]")

def format_math(example):
    return {
        "prompt": [
            {"role": "system", "content": "You are a math reasoning assistant. Think inside <think> tags, then output your answer inside <answer> tags."},
            {"role": "user", "content": example["question"]}
        ],
        "solution": example["answer"].split("#### ")[-1]
    }
train_dataset = math_data.map(format_math)

code_data = load_dataset("mbpp", split="test[:5]")

def format_code(example):
    return {
        "prompt": [
            {"role": "system", "content": "You are a coding assistant."},
            {"role": "user", "content": example["text"]}
        ],
        "solution": example["code"]
    }
test_dataset = code_data.map(format_code)

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

...

Generating train split:   0%|          | 0/374 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/90 [00:00<?, ? examples/s]

Generating prompt split:   0%|          | 0/10 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

# Train RLVR

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from rewards import format_reward_func, accuracy_reward_func

training_args = GRPOConfig(
    output_dir="./rlvr-mac-sandbox",
    learning_rate=1e-4,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    max_steps=10,
    logging_steps=2,
    bf16=True,
    use_vllm=False,
    num_generations=8,
    generation_batch_size=8,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[format_reward_func, accuracy_reward_func],
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

print("Starting Toy GRPO Training...")
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Starting Toy GRPO Training...


/Users/bui/anaconda3/envs/rlvr/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Passing `generation_config` together with generation-related arguments=({'disable_compile'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Step,Training Loss
2,0.098004
4,-0.052760
6,0.091059
8,-0.080197
10,-0.053796


TrainOutput(global_step=10, training_loss=0.00046206340193748473, metrics={'train_runtime': 216.2757, 'train_samples_per_second': 0.092, 'train_steps_per_second': 0.046, 'total_flos': 0.0, 'train_loss': 0.00046206340193748473})

# Get influence scores

In [ ]:
def get_lora_gradient(peft_model, tokenizer, prompt_text, target_text):
    """
    Extracts the flattened gradient vector of the LoRA weights for a given input/target.
    """
    peft_model.eval()
    peft_model.zero_grad()
    
    full_text = f"{prompt_text}\n{target_text}{tokenizer.eos_token}"
    inputs = tokenizer(full_text, return_tensors="pt").to(device)
    
    # Forward pass (Compute loss)
    outputs = peft_model(**inputs, labels=inputs["input_ids"])
    loss = outputs.loss
    
    # Backward pass (Calculate gradients)
    loss.backward()
    
    # Extract and flatten only the LoRA gradients
    lora_grads = []
    for name, param in peft_model.named_parameters():
        if param.requires_grad and param.grad is not None:
            lora_grads.append(param.grad.view(-1))
            
    peft_model.zero_grad() # Clean up memory
    
    # Return as one giant 1D vector
    return torch.cat(lora_grads)

# --- The Influence Calculation ---
print("Calculating Influence...")

# 1. Get the gradient for a Code Test problem
test_prompt = test_dataset[0]["prompt"][1]["content"]
test_target = test_dataset[0]["solution"]
g_test = get_lora_gradient(model, tokenizer, test_prompt, test_target)

# 2. Get the gradient for a Math Train problem
train_prompt = train_dataset[0]["prompt"][1]["content"]
train_target = train_dataset[0]["solution"]
g_train = get_lora_gradient(model, tokenizer, train_prompt, train_target)

# 3. Compute First-Order Influence (Dot Product)
influence_score = torch.dot(g_test, g_train).item()

print(f"Influence Score (Math Prompt 0 -> Code Prompt 0): {influence_score}")

Calculating Influence...
Influence Score (Math Prompt 0 -> Code Prompt 0): 0.012123634107410908
